# 04 — Market Distribution

Measure return distribution, skewness, kurtosis, quantiles and tails.


In [3]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [4]:
# Cell: Import Distribution Modules
import pandas as pd

from src.data.loaders import read_parquet, save_parquet
from src.features.returns import add_returns
from src.features.distribution import add_rolling_distribution_stats


In [5]:
# Cell: Build Distribution Features
clean = read_parquet("data/processed/clean_daily_ohlcv.parquet")
features = add_returns(clean)
features = add_rolling_distribution_stats(features, return_col="return_1d", window=60)
features.shape


(189047, 24)

In [6]:
# Cell: Summarize Distribution By Symbol
summary = (
    features.groupby("symbol").agg(
        mean_return=("return_1d", "mean"),
        median_return=("return_1d", "median"),
        skew_60d=("dist_skew_60d", "last"),
        kurt_60d=("dist_kurt_60d", "last"),
        tail_ratio_60d=("tail_ratio_60d", "last"),
    )
    .sort_values("tail_ratio_60d", ascending=False)
)
summary


,mean_return,median_return,skew_60d,kurt_60d,tail_ratio_60d
symbol,,,,,
BBC,0.003513,0.0,0.000000,-3.000000,1124.666667
ATS,0.005362,0.0,0.000000,-3.000000,17.851982
ARM,0.023143,0.0,0.479441,0.100206,13.449313
VGR,0.003568,0.0,0.603416,3.061294,2.883016
KDC,-0.000209,0.0,-0.012992,4.383120,2.712263
...,...,...,...,...,...
ART,0.000000,0.0,0.000000,-3.000000,NaN
ASA,0.000000,0.0,0.000000,-3.000000,NaN
ATB,0.000000,0.0,NaN,NaN,NaN


In [7]:
# Cell: Save Distribution Outputs
save_parquet(features, "data/processed/stock_features_distribution.parquet")
features[["symbol", "date", "return_1d", "dist_skew_60d", "dist_kurt_60d", "tail_ratio_60d"]].tail(20)


,symbol,date,return_1d,dist_skew_60d,dist_kurt_60d,tail_ratio_60d
2548,YEG,2026-06-03,-0.003319,-1.056041,4.233351,1.216713
2549,YEG,2026-06-04,-0.006659,-0.537644,3.933112,1.381191
2550,YEG,2026-06-05,-0.005587,-0.549907,3.977802,1.381191
2551,YEG,2026-06-08,-0.022472,-0.570583,4.144693,1.104487
2552,YEG,2026-06-09,-0.010345,-0.580909,4.209570,1.104487
2553,YEG,2026-06-10,0.004646,-0.580963,4.209196,1.104487
2554,YEG,2026-06-11,-0.013873,-0.549655,4.070110,1.104487
2555,YEG,2026-06-12,0.008206,-0.572692,4.318430,1.104487
2556,YEG,2026-06-15,0.011628,-0.642516,4.424935,1.104487
2557,YEG,2026-06-16,0.009195,-0.688628,4.393578,1.104487
